In [81]:
import services.utils as ut
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from nltk.stem import WordNetLemmatizer
from gensim.parsing.preprocessing import STOPWORDS
from gensim.utils import simple_preprocess
import random
import services.utils as ut
import services.model as md
import joblib
import json
np.random.seed(42)
random.seed(42)

In [82]:
def predictionoflabels(predictions, jsonfile):
    with open(jsonfile, 'r') as f:
        labels_dict = json.load(f)
    for pred in predictions:
        label_name = labels_dict.get(str(pred), "Unknown Cluster")
        print(f"Cluster {pred}: {label_name}")

In [83]:
def printlabels(jsonfile):
    with open(jsonfile, 'r') as f:
        labels_dict = json.load(f)
    for key, value in labels_dict.items():
        print(f"Cluster {key}: {value}")

In [84]:
custom_words = {
    'please', 'help', 'assist', 'support', 'thanks', 'thank','soon','mentioned',
    'im', 'ive', 'us','would', 'could', 'need', 'want', 'trying',
    'tried','check', 'checked', 'make', 'made', 'get', 'getting','also',
    'use', 'using', 'used','thing', 'something', 'anything', 'everything',
    'way', 'time','issue', 'problem', 'request', 'work', 'working', 'fine',
    'available', 'recent', 'recently','facing', 'doe', 'noticed', 'happening',
    'started', 'happen','different', 'steps', 'did', 'regards','already', 'multiple',
    'last','times','followed', 'reviewed','specific', 'possible', 'related','new',
    'old','find', 'try', 'say', 'mean','name', 'email', 'price', 'one', 'unresolved',
    'add','note', 'may', 'dont', 'know','sure', 'changes', 'performed', 'properly',
    'original','like', 'similar','reported','doesnt', 'sometimes', 'acts', 'works',
    'ensure', 'desired', 'action', 'remains', 'life', 'seems', 'might', 'guide',
    'much', 'others', 'heavily', 'daily', 'task', 'affecting', 'assistance','hoping',
    'persists','didnt','option', 'perform', 'recommendation', 'information', 'official',
    'solution', 'provide', 'making', 'user', 'customer', 'item', 'device','far', 'luck',
    'contact', 'contacted', 'occurring','resolve', 'function', 'came', 'having', 'change',
    'haven', 'let', 'unable', 'able', 'afterward', 'var', 'step', 'order'
}



In [85]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\{.*?\}|\[.*?\]|<.*?>|\(.*?\)', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text) # remove URLs
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', ' ', text) # remove email addresses
    text = re.sub(r'[^\x00-\x7F]+', ' ', text) # remove non ASCII characters
    text = re.sub(r'[^a-z\s]', '', text) 
    text = re.sub(r'\b[a-zA-Z]{1,2}\b', '', text)  # remove very short words
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [86]:
lemmatizer = WordNetLemmatizer()
final_stopwords = STOPWORDS.union(custom_words)
custom_words_lemma = set([lemmatizer.lemmatize(w.lower()) for w in final_stopwords])

def preprocess(text):
    text = str(text).lower()
    tokens = simple_preprocess(text, deacc=True)
    processed_tokens = []
    for word in tokens:
        lemma = lemmatizer.lemmatize(word)
        if (lemma not in custom_words_lemma and len(lemma) > 2 and lemma.isalnum()):
            processed_tokens.append(lemma)
    
    return " ".join(processed_tokens)

In [87]:
def nlp_cleaner(text_list):
    results = []
    for text in text_list:
        cleaned = clean_text(text) 
        processed = preprocess(cleaned)
        results.append(processed)
    return results

In [88]:
def clean_for_embeddings(text):
    text = text.lower()
    text = re.sub(r'\{.*?\}|\[.*?\]|<.*?>|\(.*?\)', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text) # remove URLs
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', ' ', text) # remove email addresses
    text = re.sub(r'[^\x00-\x7F]+', ' ', text) # remove non-ASCII characters
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [89]:
def nlp_cleaner_embeddings(text_list):
    results = []
    for text in text_list:
        cleaned = clean_for_embeddings(text) 
        results.append(cleaned)
    return results

In [90]:
raw_data = ["I need help with my account setup", "Software is crashing in the production environment"]

## Kmeans

### Kmean TFIDF without SVD

In [91]:
Kmeans_tfidf_without_svd_prediction_file = ut.loadingfile('kmeans', 'kmeanstfidfwithoutsvd_cluster_labels.json')
printlabels(Kmeans_tfidf_without_svd_prediction_file)

Cluster 0: Firmware Stability & Freezes
Cluster 1: General Productivity & Power Config
Cluster 2: Accidental File Deletion & Recovery
Cluster 3: Software Version & Crash Reports
Cluster 4: Unexpected UI Error Messages
Cluster 5: Account Security & Password Recovery
Cluster 6: Security Privacy & Screen Flickering
Cluster 7: Intermittent Performance & Returns
Cluster 8: App Cache & Maintenance
Cluster 9: Community Forum & Online Tutorials
Cluster 10: Hardware Noise & Fault Suspicions
Cluster 11: Manual & Website Configuration
Cluster 12: Productivity Impact & Reliability
Cluster 13: Factory Reset & Connection Failures
Cluster 14: Network & Home WiFi Stability
Cluster 15: Battery Degradation
Cluster 16: Cables & Peripheral Accessories


In [92]:
Kmeans_tfidf_without_svd = ut.load_model('kmeans', 'kmeans_tfidf_without_pca.joblib')
predictions = Kmeans_tfidf_without_svd.predict(raw_data)
predictionoflabels(predictions, Kmeans_tfidf_without_svd_prediction_file)

Cluster 1: General Productivity & Power Config
Cluster 3: Software Version & Crash Reports


### Kmean TFIDF with SVD

In [93]:
kmeans_tfidf_svd_prediction_file = ut.loadingfile('kmeans', 'kmeanstfidfsvd_cluster_labels.json')
printlabels(kmeans_tfidf_svd_prediction_file)

Cluster 0: Account Display & Access
Cluster 1: Admin & Account Locking
Cluster 2: Application Stability & Battery
Cluster 3: Account Recovery History
Cluster 4: Mobile Billing & Android Support
Cluster 5: Factory Resets & Apple Ecosystem
Cluster 6: Software Permissions & Settings
Cluster 7: General Account Support (High Volume)
Cluster 8: Address & Location Management
Cluster 9: Troubleshooting Feedback
Cluster 10: Automated Account Syncing
Cluster 11: Online Profile Authorization
Cluster 12: Software Productivity Profiles
Cluster 13: General Account Inquiries
Cluster 14: System Setup & Initialization
Cluster 15: Peripheral & Adapter Connectivity
Cluster 16: Order History & Shipping
Cluster 17: Third-Party Integration
Cluster 18: Cloud Services & Billing Address
Cluster 19: Security Credentials & Lockouts


In [94]:
kmeans_tfidf_svd = ut.load_model('kmeans', 'kmeans_tfidf_with_svd.joblib')
predictions = kmeans_tfidf_svd.predict(raw_data)
predictionoflabels(predictions, kmeans_tfidf_svd_prediction_file)

Cluster 1: Admin & Account Locking
Cluster 11: Online Profile Authorization


### Kmean Embeddings without PCA

In [95]:
kmeans_embeddings_without_pca_prediction_file = ut.loadingfile('kmeans', 'kmeansembeddingswithoutpca_cluster_labels.json')
printlabels(kmeans_embeddings_without_pca_prediction_file)

Cluster 0: Intermittent Performance & Battery Drain
Cluster 1: Data Privacy & Security Concerns
Cluster 2: Data Recovery & Hardware Replacement (Urgent)
Cluster 3: Account Lockout & Password Reset Failures
Cluster 4: Software Bugs & Application Crashes
Cluster 5: Firmware Updates & Consistently Occurring Issues
Cluster 6: Power Failures & Support Responsiveness
Cluster 7: Credential Errors & Widespread Device Issues
Cluster 8: UI Error Messages & Screen Pop-ups
Cluster 9: WiFi Connectivity & Network Detection
Cluster 10: Screen Flickering & Hardware Troubleshooting
Cluster 11: Charging Failures & Original Accessories
Cluster 12: Network Setup & App Persistence Issues
Cluster 13: Manual Configuration & Factory Resets
Cluster 14: Version Updates & Intermittent Glitches
Cluster 15: Strange Noises & Suspected Hardware Defects
Cluster 16: Purchasing Assistance & Navigation Help
Cluster 17: Software Freezing & Security Stability
Cluster 18: Stable Internet & Disconnection Troubleshooting


In [96]:
kmeans_embeddings_without_pca = ut.load_model('kmeans', 'kmeans_embeddings_without_pca.joblib')
predictions = kmeans_embeddings_without_pca.predict(raw_data)
predictionoflabels(predictions, kmeans_embeddings_without_pca_prediction_file)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Cluster 7: Credential Errors & Widespread Device Issues
Cluster 4: Software Bugs & Application Crashes


### Kmean Embeddings with PCA

In [97]:
kmeans_embeddings_with_pca_prediction_file = ut.loadingfile('kmeans', 'kmeansembeddingswithpca_cluster_labels.json')
printlabels(kmeans_embeddings_with_pca_prediction_file)

Cluster 0: Account Recovery & Invalid Credentials
Cluster 1: System Freezes & Error Pop-ups
Cluster 2: Power-On Failures & Boot Issues
Cluster 3: Factory Reset & Persistent Product Issues
Cluster 4: Navigation Support & Action Guidance
Cluster 5: Intermittent Performance & Battery Drain
Cluster 6: Configuration & Manual Troubleshooting
Cluster 7: Data Recovery & Syncing Conflicts
Cluster 8: Home Wi-Fi Network Detection
Cluster 9: Application Crashes & Software Bugs
Cluster 10: Data Security & Privacy Concerns
Cluster 11: Charging Failures & Power Supply
Cluster 12: Screen Flickering & Display Hardware
Cluster 13: Locked Accounts & Browser Authentication
Cluster 14: Firmware Update Post-Install Issues
Cluster 15: Initial Network Setup Failures
Cluster 16: Intermittent Internet Disconnections
Cluster 17: Mechanical Noises & Hardware Defects
Cluster 18: Support Escalation & Reset Failures


In [98]:
kmeans_embeddings_with_pca = ut.load_model('kmeans', 'kmeans_embeddings_with_pca.joblib')
predictions = kmeans_embeddings_with_pca.predict(raw_data)
predictionoflabels(predictions, kmeans_embeddings_with_pca_prediction_file)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Cluster 10: Data Security & Privacy Concerns
Cluster 9: Application Crashes & Software Bugs


## HAC

### HAC without TFIDF

In [99]:
hac_tfidf_without_svd_prediction_file = ut.loadingfile('hac', 'HACtfidfwithoutsvd_cluster_labels.json')
printlabels(hac_tfidf_without_svd_prediction_file)

Cluster 0: General Maintenance & Product Inquiries
Cluster 1: Firmware & Software Version Updates
Cluster 2: Account Recovery & Password Reset
Cluster 3: Mechanical Noise & Hardware Suspicions
Cluster 4: Screen Pop-ups & UI Error Messages
Cluster 5: Network Connection Failures
Cluster 6: Intermittent Internet & Router Stability
Cluster 7: File Retrieval & Document Loss
Cluster 8: Critical Power-On & Boot Failures
Cluster 9: Application Crashes & Version Errors
Cluster 10: Accidental Deletion & Data Recovery
Cluster 11: Battery Degradation & Life Cycle
Cluster 12: System Crash & Data Loss Recovery
Cluster 13: Invalid Login Credentials & Access
Cluster 14: Software Freezing & Glitch Stability
Cluster 15: Home Wi-Fi Detection & Setup
Cluster 16: Hardware Repair & Replacement Requests
Cluster 17: Installation Errors & Unexpected Loss
Cluster 18: Data Security & Privacy Concerns


### HAC with TFIDF

In [100]:
hac_tfidf_without_svd_prediction_file = ut.loadingfile('hac', 'HACtfidfwithsvd_pipeline_cluster_labels.json')
printlabels(hac_tfidf_without_svd_prediction_file)

Cluster 0: Software Glitches & Stability Issues
Cluster 1: General Maintenance & Product Inquiries
Cluster 2: Account Recovery & Password Reset
Cluster 3: General Network Connection Failures
Cluster 4: Mechanical Noise & Hardware Suspicions
Cluster 5: Application Crashes & Version Errors
Cluster 6: Firmware Updates & Shipping Inquiries
Cluster 7: Intermittent Internet & Router Stability
Cluster 8: UI Error Messages & Screen Flickering
Cluster 9: Data Security & Refund Concerns
Cluster 10: Accidental Deletion & Urgent Recovery
Cluster 11: Battery Performance & Power Issues
Cluster 12: File Retrieval & Document Loss
Cluster 13: Invalid Login Credentials & Access
Cluster 14: System Crash & Data Loss Recovery
Cluster 15: Home Wi-Fi Detection & Setup
Cluster 16: Installation Errors & Unexpected Loss
Cluster 17: Hardware Repair & Replacement Requests


### HAC with Embeddings

In [101]:
hac_embeddings_without_pca_prediction_file = ut.loadingfile('hac', 'HACembeddingswithoutpca_pipeline_cluster_labels.json')
printlabels(hac_embeddings_without_pca_prediction_file)

Cluster 0: Network Connectivity & Battery Performance
Cluster 1: Firmware Updates & Factory Reset Issues
Cluster 2: Intermittent Hardware Noise & Error Messages
Cluster 3: Urgent Data Recovery & Accidental Deletion
Cluster 4: Account Access & Locked Credentials
Cluster 5: Unstable Internet & Intermittent Disconnection
Cluster 6: General Product Issue & Data Security Concerns
Cluster 7: Productivity Impact & Battery Life Decrease
Cluster 8: UI Navigation & Cart Instructions
Cluster 9: Account Verification & Personal Information
Cluster 10: Software Update & Installation Errors
Cluster 11: Display Flickering & UI Pop-up Messages
Cluster 12: Hardware Repair & Replacement Inquiries
Cluster 13: Login Credential Failures & Website Support
Cluster 14: System Crash & Post-Crash Recovery
Cluster 15: Wireless Setup & Home Wi-Fi Detection
Cluster 16: Purchase Refunds & Warranty Inquiries
Cluster 17: Feature-Specific Stability & Application Errors
Cluster 18: General Troubleshooting & Update Verif

### HAC with Embeddings PCA

In [102]:
hac_embeddings_with_pca_prediction_file = ut.loadingfile('hac', 'HACembeddingswithpca_cluster_labels.json')
printlabels(hac_embeddings_with_pca_prediction_file)

Cluster 0: Network Setup & Data Security Inquiries
Cluster 1: Urgent Data Recovery & Accidental Deletion
Cluster 2: Account Access & Locked Credentials
Cluster 3: General Troubleshooting & Factory Reset
Cluster 4: Firmware Updates & Post-Update Issues
Cluster 5: Productivity Impact & Battery Life Decrease
Cluster 6: Intermittent System Stability & Manual Steps
Cluster 7: Software Bug Reporting & Data Loss
Cluster 8: UI Navigation & Online Cart Guidance
Cluster 9: Critical Power Failure & Boot Issues
Cluster 10: Charger Compatibility & Power Supply Problems
Cluster 11: Screen Flickering & Hardware Display Problems
Cluster 12: Internet Disconnection & ISP Troubleshooting
Cluster 13: Screen Pop-ups & Peculiar Error Messages
Cluster 14: Intermittent Mechanical Noise & Hardware Suspicions
Cluster 15: System Freezing & Software Stability Glitches
Cluster 16: Password Recovery & Reset Link Failures
Cluster 17: Application Crashes & Version Fix Requests
Cluster 18: General Security Verificatio

## LDA

### LDA with TFIDF

In [103]:
ladtfidf_prediction_file = ut.loadingfile('lda', 'LDAtfidf_pipeline_cluster_labels.json')
printlabels(ladtfidf_prediction_file)

Cluster 0: Widespread Data Loss & Model Response Issues
Cluster 1: Community Tutorials & Online Troubleshooting
Cluster 2: Security Concerns & Factory Reset Safety
Cluster 3: Network Connectivity & Productivity Impact
Cluster 4: Firmware Updates & Document Retrieval
Cluster 5: Home Wi-Fi & Charger Connection Troubleshooting
Cluster 6: Account Login & Credential Recovery
Cluster 7: Battery Degradation & Hardware Noise Symptoms
Cluster 8: Screen Error Messages & Pop-up Configurations
Cluster 9: Hardware Repair & Replacement Inquiries
Cluster 10: Intermittent Feature Failures & Settings
Cluster 11: App Software Bugs & Cache Management
Cluster 12: Software Version Updates & Maintenance
Cluster 13: Product Manuals & Account Password Reset


In [104]:
ladtfidf = ut.load_model('lda', 'lda_with_tfidf.joblib')
predictionsldatfidf = ladtfidf.transform(raw_data)
predictions_final_ldatfidf = np.argmax(predictionsldatfidf, axis=1)
predictionoflabels(predictions_final_ldatfidf, ladtfidf_prediction_file)

Cluster 1: Community Tutorials & Online Troubleshooting
Cluster 12: Software Version Updates & Maintenance


### LDA with countvectorizer

In [105]:
ldacountvectorizer_prediction_file = ut.loadingfile('lda', 'ladwithcountvectorizer_pipeline_cluster_labels.json')
printlabels(ldacountvectorizer_prediction_file)

Cluster 0: Widespread Data Loss & Model Failures
Cluster 1: Account Lockout & Unlock Assistance
Cluster 2: Data Security & Privacy Concerns
Cluster 3: Software Updates & Application Crashes
Cluster 4: Productivity Impact & Battery Degradation
Cluster 5: Firmware Updates & Home Wi-Fi Connectivity
Cluster 6: Invalid Login Credentials & Access Recovery
Cluster 7: Hardware Noise & Technical Support Site
Cluster 8: Screen Error Messages & Pop-up Configuration
Cluster 9: Hardware Repair & Replacement Requests
Cluster 10: Intermittent Feature & Settings Configuration
Cluster 11: App Cache Management & Software Bugs
Cluster 12: Product Purchase & Charger Performance
Cluster 13: Factory Reset & Password Recovery
Cluster 14: Community Forums & Peripheral Troubleshooting


In [106]:
ldacountvectorizer = ut.load_model('lda', 'lda_with_count_vectorizer.joblib')
ldacountvectorizer_prediction_file = ut.loadingfile('lda', 'ladwithcountvectorizer_pipeline_cluster_labels.json')
predictionsldacountvectorizer = ldacountvectorizer.transform(raw_data)
predictions_final_ldacountvectorizer = np.argmax(predictionsldacountvectorizer, axis=1)
predictionoflabels(predictions_final_ldacountvectorizer, ldacountvectorizer_prediction_file)

Cluster 1: Account Lockout & Unlock Assistance
Cluster 3: Software Updates & Application Crashes


### LDA with Gensim Topics

In [107]:
def predict_gensim_lda(model, dictionary, raw_docs):
    tokenized_data = [str(doc).split() for doc in raw_docs]
    bow_corpus = [dictionary.doc2bow(text) for text in tokenized_data]
    final_labels = []
    for bow in bow_corpus:
        topic_probs = model.get_document_topics(bow, minimum_probability=0.0)
        best_topic = max(topic_probs, key=lambda x: x[1])[0]
        final_labels.append(best_topic)
        
    return np.array(final_labels)

In [108]:
ldagensim_prediction_file = ut.loadingfile('lda', 'lda_gensim_topics_pipeline_cluster_labels.json')
printlabels(ldagensim_prediction_file)

Cluster 0: Software Version & Payment Inquiries
Cluster 1: Network Setup & System Freezing
Cluster 2: Hardware Noise & Productivity Hindrance
Cluster 3: Product Purchase & Cart Transactions
Cluster 4: Login Credentials & Screen Error Messages
Cluster 5: Intermittent Data Loss & Warranty Status
Cluster 6: Data Security & Hardware Repair Concerns
Cluster 7: Battery Life Decrease & Refund Inquiries
Cluster 8: Wi-Fi Connectivity & Manual Troubleshooting
Cluster 9: Adapters, Cables & Peripheral Support
Cluster 10: Factory Reset & Community Tutorials
Cluster 11: Firmware Updates & Checkout Issues
Cluster 12: App Cache Clearing & Unexpected Errors
Cluster 13: Application Crashes & Feature Bugs
Cluster 14: Internet Connection & Router Stability
Cluster 15: Account Configuration & Security Lockouts
Cluster 16: Accidental Deletion & Urgent Data Recovery
Cluster 17: Screen Flickering & Charging Hardware


In [109]:
ldagensim = ut.load_model('lda', 'lda_gensim_topics.joblib')
dictionarygensim = ut.load_model('lda', 'lda_gensim_topics_dict.joblib')
ldagensim_prediction_file = ut.loadingfile('lda', 'lda_gensim_topics_pipeline_cluster_labels.json')
predictions_final = predict_gensim_lda(ldagensim, dictionarygensim, raw_data)
predictionoflabels(predictions_final, ldagensim_prediction_file)

Cluster 16: Accidental Deletion & Urgent Data Recovery
Cluster 0: Software Version & Payment Inquiries
